# 05.03_OrthoFinder_results_plot_R

整理/绘制 OrthoFinder 结果。

- 当前文件：`analysis/05_genome_analysis/05.03_OrthoFinder_results_plot_R.ipynb`
- 原始来源：`Codes/05.03_R_OrthoFinder_results_plot.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`ComplexUpset`, `UpSetR`, `ggplot2`, `tidyverse`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


In [ ]:
fig_dir <- '/share/home/zhangze/zz/NeuralOrigin/Figures'

ZZ：59绘制UpSet

In [ ]:
# 1. 安装必要的R包（如果你还没有安装的话，取消下面两行的注释）
# install.packages("UpSetR")
# install.packages("tidyverse")

library(UpSetR)
library(tidyverse)

In [ ]:
# 读取原始GeneCount数据
gc <- read.delim("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step3_OrthogroupsGeneCount/Orthogroups.GeneCount.tsv", row.names = 1)
# 去掉Total列
gc <- gc[, !colnames(gc) %in% "Total"]
# 确认9个物种列
species_cols <- c("Auco.protein", "ClH23.protein", "Clhe.protein", "Dare.protein", 
                  "HoH13.protein", "Neve.protein", "Spla.protein", "TrH1.protein", "TrH2.protein")
gc <- gc[, species_cols]
# 标记全共享（每个物种计数>0）
all_present <- apply(gc, 1, function(x) all(x > 0))
sum(all_present)  # 应为3339

# 导出这些orthogroup的ID
diff_ogs <- rownames(gc)[all_present]
writeLines(diff_ogs, "/share/home/zhangze/zz/NeuralOrigin/Figures/59.my_all_present_ogs.txt")

In [ ]:
# 2. 读取数据 
# 请将文件路径替换为你电脑上实际的 Orthogroups.GeneCount.tsv 路径
gene_count_data <- read.delim("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step3_OrthogroupsGeneCount/Orthogroups.GeneCount.tsv", header = TRUE, stringsAsFactors = FALSE)

# 3. 数据预处理：转换为 存在(1) / 缺失(0) 矩阵
# 去除 "Orthogroup" (第一列) 和 "Total" (最后一列)
species_matrix <- gene_count_data %>%
  select(-Orthogroup, -Total) %>%  
  # 如果某个物种在该同源群有基因(数量>0)，记为1；否则记为0
  mutate(across(everything(), ~ ifelse(. > 0, 1, 0))) 

# 为了确保列名准确，你可以提取你表格中的这9个物种的列名：
# Auco.protein, ClH23.protein, Clhe.protein, Dare.protein, HoH13.protein, Neve.protein, Spla.protein, TrH1.protein, TrH2.protein
species_names <- colnames(species_matrix)

# 4. 绘制高颜值的 UpSet 图
upset_plot <- upset(
  species_matrix,
  nsets = 9,                     # 展示你所有的 9 个物种
  nintersects = 15,              # 展示交集数量排名前40的组合（可根据你的画幅大小调整）
  sets = species_names,          # 使用提取的物种名称
  keep.order = FALSE,            # 自动按照物种总同源群数量进行排序
  order.by = "freq",             # 顶部的柱子按照交集大小（包含的同源群数量）从大到小排序
  
  # --- 自定义标签和颜色（方便放入毕业论文） ---
  mainbar.y.label = "Number of Shared Orthogroups", # 顶部柱状图的Y轴标签
  sets.x.label = "Total Orthogroups per Species",   # 左侧条形图的X轴标签
  
  # 调整字体大小，使图在A4纸上打印也清晰可见
  # 顺序为: 顶部Y轴标题, 顶部Y轴刻度, 左侧X轴标题, 左侧X轴刻度, 物种名称, 柱子上的数字
  text.scale = c(1.5, 1.2, 1.5, 1.2, 1.3, 1), 
  
  # 颜色搭配
  matrix.color = "#2c3e50",      # 底部点和连线的颜色 (深蓝灰)
  main.bar.color = "#34495e",    # 顶部柱子的默认颜色
  sets.bar.color = "#7f8c8d"     # 左侧各个物种总数柱子的颜色
)

# 查看图片
print(upset_plot)

# 5. 保存为高清 PDF 和 PNG 格式，方便插入论文 (8 x 6 英寸比例通常较好)
pdf(paste0(fig_dir,'/59.UpSetPlot_Orthogroups.pdf'), width = 10, height = 6, onefile = FALSE)
print(upset_plot)
dev.off()

png(paste0(fig_dir,'/59.UpSetPlot_Orthogroups.png'), width = 10, height = 6, units = "in", res = 300)
print(upset_plot)
dev.off()

In [ ]:
# 计算全共享（每个物种>0）
all_present_my <- apply(species_matrix, 1, function(x) all(x == 1))
sum(all_present_my)  # 应为3339

# 读取OrthoFinder官方全共享列表（从Orthogroups.tsv中提取）
# 注意：官方列表通常记录在 Statistics_Overall.tsv 中，但未给出具体ID。
# 替代方案：检查 GeneCount.tsv 中是否有某些orthogroup在某个物种中计数为0，但官方认为存在？

In [ ]:
# 假设您已经执行了前面的代码，species_matrix 已存在于环境中

# 1. 为每个OG生成一个模式字符串（例如 "101010101"）
pattern <- apply(species_matrix, 1, function(x) paste(x, collapse = ""))

# 2. 统计每种模式出现的频次
pattern_counts <- table(pattern)

# 3. 将所有组合情况转换为数据框
all_combinations <- data.frame(
  Pattern = names(pattern_counts),
  Count = as.numeric(pattern_counts),
  stringsAsFactors = FALSE
)

# 按频次降序排列
all_combinations <- all_combinations[order(-all_combinations$Count), ]

# 添加物种组合说明（可选）
species_abbr <- c("Auco", "ClH23", "Clhe", "Dare", "HoH13", "Neve", "Spla", "TrH1", "TrH2")
all_combinations$SpeciesSet <- sapply(all_combinations$Pattern, function(p) {
  idx <- which(strsplit(p, "")[[1]] == "1")
  if(length(idx) == 0) return("None")
  paste(species_abbr[idx], collapse = ",")
})

# 打印全部组合情况
cat("========== 全部组合情况（共", nrow(all_combinations), "种组合）==========\n")
print(all_combinations)

# 4. 统计9个物种全共享的OG数量
full_pattern <- paste(rep("1", 9), collapse = "")  # "111111111"
full_shared_count <- ifelse(full_pattern %in% names(pattern_counts), pattern_counts[full_pattern], 0)
cat("\n========== 9个物种全共享的OG数量 ==========\n")
cat(full_shared_count, "\n")

# 5. 统计每个物种的特异性OG数量（仅在该物种中存在）
species_names <- colnames(species_matrix)
species_specific_counts <- c()

for (i in 1:length(species_names)) {
  # 生成只有第i位为1，其余为0的模式
  spec_pattern <- paste(c(rep("0", i-1), "1", rep("0", length(species_names)-i)), collapse = "")
  count <- ifelse(spec_pattern %in% names(pattern_counts), pattern_counts[spec_pattern], 0)
  species_specific_counts <- c(species_specific_counts, count)
}

names(species_specific_counts) <- species_names

cat("\n========== 每个物种的特异性OG数量 ==========\n")
print(species_specific_counts)

# 可选：保存为CSV文件
write.csv(all_combinations, "all_orthogroup_combinations.csv", row.names = FALSE)

specific_df <- data.frame(
  Species = species_names,
  Specific_OG_Count = species_specific_counts
)
write.csv(specific_df, "species_specific_OG_counts.csv", row.names = FALSE)

In [ ]:
# 1. 安装必要的R包（如果没有安装的话，取消下面两行的注释）
# install.packages("ComplexUpset")
# install.packages("ggplot2")
# install.packages("tidyverse")

library(ComplexUpset)
library(ggplot2)
library(tidyverse)

# 2. 读取数据
# 请将文件路径替换为你电脑上实际的 Orthogroups.GeneCount.tsv 路径
gene_count_data <- read.delim("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step3_OrthogroupsGeneCount/Orthogroups.GeneCount.tsv", header = TRUE, stringsAsFactors = FALSE)

# 3. 数据预处理
# 注意：ComplexUpset 更倾向于使用 逻辑值(TRUE/FALSE) 来表示存在与否
species_matrix <- gene_count_data %>%
  select(-Orthogroup, -Total) %>%  
  # 将大于0的值转化为 TRUE (存在)，等于0的值转化为 FALSE (缺失)
  mutate(across(everything(), ~ . > 0)) 

species_names <- colnames(species_matrix)

# 4. 绘制高颜值的 UpSet 图 (带具体数值)
upset_plot <- upset(
  species_matrix,
  species_names,
  n_intersections = 15,               # 展示排名前40的交集组合
  name = "Species",                   # 左下角的坐标轴名称
  width_ratio = 0.25,                 # 调整左侧柱状图与右侧主体图的宽度比例
  
  # --- 关键配置：自定义顶部的柱状图 (交集大小) ---
  base_annotations = list(
    'Intersection size' = intersection_size(
      counts = TRUE,                  # 在顶部柱子上显示具体的交集数值
      text = list(size = 3.5, vjust = -0.5) # 调整顶部数字的大小和位置
    ) + 
    ylab("Number of Shared Orthogroups") +
    theme(
      axis.title.y = element_text(size = 12),
      axis.text.y = element_text(size = 10)
    )
  ),
  
  # --- 关键配置：自定义左侧的柱状图 (物种同源群总数) ---
  set_sizes = (
    upset_set_size() + 
    # 使用 geom_text 在柱子末端添加具体数值
    geom_text(
      aes(label = after_stat(count)), 
      stat = 'count', 
      hjust = -0.15,                  # 将数字向右微调，避免与柱体顶端重叠
      size = 3.5                      # 调整左侧数字的大小
    ) +
    ylab("Total Orthogroups per Species") +
    # 扩展 Y 轴（在水平图中表现为X轴的长度），防止数字被边缘裁掉
    # 这里的 11000 是根据你数据中最大的 orthogroups 数量 (ClH23: 8102 ~ HoH13: 11320 等) 估算的，你可以根据实际出图效果调大
    expand_limits(y = 13000) +
    theme(
      axis.title.x = element_text(size = 12),
      axis.text.x = element_text(size = 10)
    )
  ),
  
  # --- 整体主题微调 ---
  themes = upset_modify_themes(
    list(
      'intersections_matrix' = theme(text = element_text(size = 12))
    )
  )
)

# 查看图片
print(upset_plot)

# # 5. 保存为高清 PDF 和 PNG
# pdf("Orthogroups_UpSetPlot_with_numbers.pdf", width = 11, height = 7, onefile = FALSE)
# print(upset_plot)
# dev.off()

# png("Orthogroups_UpSetPlot_with_numbers.png", width = 11, height = 7, units = "in", res = 300)
# print(upset_plot)
# dev.off()